# Bitrate vs Quality Summary

This notebook reads the aggregated 0428 metrics CSV, groups results by scheme and budget, and plots mean PSNR / SSIM with a shaded ±1 std band across cameras.

In [ ]:
from __future__ import annotations

import csv
import os
from collections import defaultdict
from pathlib import Path
from statistics import fmean, pstdev

import matplotlib.pyplot as plt

OUTPUT_ROOT = Path(os.environ.get(
    "OUTPUT_ROOT",
    "/mnt/data1/samk/gs-quic/cs5262_tile_quic/dlapisgs-utility/output/0428",
))
SUMMARY_CSV = OUTPUT_ROOT / "metrics" / "summary.csv"
PLOT_DIR = OUTPUT_ROOT / "metrics" / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
def load_rows(csv_path: Path) -> list[dict[str, str]]:
    if not csv_path.exists():
        raise FileNotFoundError(f"Metrics CSV not found: {csv_path}")

    with csv_path.open(newline="", encoding="utf-8") as fp:
        return list(csv.DictReader(fp))


def group_by_scheme_and_budget(rows: list[dict[str, str]]) -> dict[str, dict[float, list[dict[str, str]]]]:
    grouped: dict[str, dict[float, list[dict[str, str]]]] = defaultdict(lambda: defaultdict(list))
    for row in rows:
        scheme = row["scheme"]
        budget_mb = float(row["budget_mb"])
        grouped[scheme][budget_mb].append(row)
    return grouped


def summarize(grouped: dict[str, dict[float, list[dict[str, str]]]], metric_key: str) -> dict[str, list[dict[str, float]]]:
    summary: dict[str, list[dict[str, float]]] = {}
    for scheme, budgets in grouped.items():
        scheme_summary: list[dict[str, float]] = []
        for budget_mb in sorted(budgets):
            values = [float(row[metric_key]) for row in budgets[budget_mb]]
            scheme_summary.append({
                "budget_mb": budget_mb,
                "mean": fmean(values),
                "std": pstdev(values) if len(values) > 1 else 0.0,
            })
        summary[scheme] = scheme_summary
    return summary


def plot_summary(summary: dict[str, list[dict[str, float]]], metric_name: str, output_prefix: Path) -> None:
    fig, ax = plt.subplots(figsize=(8.5, 5.2), constrained_layout=True)
    for scheme, points in summary.items():
        budgets = [point["budget_mb"] for point in points]
        means = [point["mean"] for point in points]
        stds = [point["std"] for point in points]
        ax.plot(budgets, means, marker="o", linewidth=2, label=scheme)
        lower = [mean - std for mean, std in zip(means, stds)]
        upper = [mean + std for mean, std in zip(means, stds)]
        ax.fill_between(budgets, lower, upper, alpha=0.18)

    ax.set_xscale("log")
    ax.set_xlabel("Budget (MB)")
    ax.set_ylabel(metric_name)
    ax.set_title(f"0428 {metric_name} vs bitrate")
    ax.legend(title="Scheme")

    png_path = output_prefix.with_suffix(".png")
    pdf_path = output_prefix.with_suffix(".pdf")
    fig.savefig(png_path, dpi=300)
    fig.savefig(pdf_path)
    plt.show()
    print(f"Saved {png_path}")
    print(f"Saved {pdf_path}")

In [ ]:
rows = load_rows(SUMMARY_CSV)
grouped = group_by_scheme_and_budget(rows)

psnr_summary = summarize(grouped, "psnr_mean")
ssim_summary = summarize(grouped, "ssim_mean")

plot_summary(psnr_summary, "PSNR", PLOT_DIR / "psnr_vs_bitrate")
plot_summary(ssim_summary, "SSIM", PLOT_DIR / "ssim_vs_bitrate")